In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

print("Libraries loaded!")

Libraries loaded!


In [2]:
df = pd.read_csv('../data/IPL.csv', low_memory=False)

# Useful columns lo
cols = ['match_id', 'batting_team', 'bowling_team',
        'toss_winner', 'toss_decision', 'venue', 
        'year', 'match_won_by', 'win_outcome']

match_df = df[cols].copy()

# Match level pe lao
match_df = match_df.drop_duplicates(subset='match_id')

print("Shape:", match_df.shape)

Shape: (1193, 9)


In [3]:
# Step 1 — No result matches hatao
match_df = match_df[match_df['win_outcome'].notna()]
match_df = match_df[match_df['match_won_by'] != 'Unknown']

# Step 2 — Spelling fix karo
fix_teams = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

for col in ['batting_team', 'bowling_team', 'toss_winner', 'match_won_by']:
    match_df[col] = match_df[col].replace(fix_teams)

# Step 3 — Columns rename karo
match_df = match_df.rename(columns={
    'batting_team': 'team1',
    'bowling_team': 'team2'
})

# Step 4 — Target variable banao
match_df['team1_win'] = (match_df['match_won_by'] == match_df['team1']).astype(int)

print("Shape after cleaning:", match_df.shape)
print("\nFinal columns:", match_df.columns.tolist())
print("\nSample:")
match_df[['team1', 'team2', 'match_won_by', 'team1_win']].head()

Shape after cleaning: (1169, 10)

Final columns: ['match_id', 'team1', 'team2', 'toss_winner', 'toss_decision', 'venue', 'year', 'match_won_by', 'win_outcome', 'team1_win']

Sample:


,team1,team2,match_won_by,team1_win
0,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,1
225,Chennai Super Kings,Punjab Kings,Chennai Super Kings,1
473,Rajasthan Royals,Delhi Capitals,Delhi Capitals,0
692,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bengaluru,0
938,Deccan Chargers,Kolkata Knight Riders,Kolkata Knight Riders,0


In [4]:
# Feature 1 — Venue win rate for team1
venue_win_rate = match_df.groupby(
    ['venue', 'team1'])['team1_win'].mean().reset_index()
venue_win_rate.columns = ['venue', 'team1', 'venue_win_rate']

match_df = match_df.merge(venue_win_rate, on=['venue', 'team1'], how='left')
match_df['venue_win_rate'] = match_df['venue_win_rate'].fillna(0.5)

print("Venue win rate added!")
print(match_df[['team1', 'venue', 'venue_win_rate']].head())

Venue win rate added!
                   team1                                       venue  \
0  Kolkata Knight Riders                       M Chinnaswamy Stadium   
1    Chennai Super Kings  Punjab Cricket Association Stadium, Mohali   
2       Rajasthan Royals                            Feroz Shah Kotla   
3         Mumbai Indians                            Wankhede Stadium   
4        Deccan Chargers                                Eden Gardens   

   venue_win_rate  
0        0.285714  
1        0.500000  
2        0.500000  
3        0.611111  
4        0.000000  


In [5]:
# Feature 2 & 3 — Last 5 matches win rate
# match_id se order maintain karo
match_df = match_df.sort_values('match_id').reset_index(drop=True)

def recent_form(df, team_col, result_col, window=5):
    return df.groupby(team_col)[result_col].transform(
        lambda x: x.shift(1).rolling(window, min_periods=1).mean()
    )

match_df['team1_recent_form'] = recent_form(match_df, 'team1', 'team1_win')
match_df['team2_recent_form'] = 1 - recent_form(match_df, 'team2', 'team1_win')

# NaN ko 0.5 se fill karo — neutral value
match_df['team1_recent_form'] = match_df['team1_recent_form'].fillna(0.5)
match_df['team2_recent_form'] = match_df['team2_recent_form'].fillna(0.5)

print("Recent form added!")
print(match_df[['team1', 'team2', 'team1_recent_form', 'team2_recent_form']].head(10))

Recent form added!
                   team1                        team2  team1_recent_form  \
0  Kolkata Knight Riders  Royal Challengers Bengaluru                0.5   
1    Chennai Super Kings                 Punjab Kings                0.5   
2       Rajasthan Royals               Delhi Capitals                0.5   
3         Mumbai Indians  Royal Challengers Bengaluru                0.5   
4        Deccan Chargers        Kolkata Knight Riders                0.5   
5           Punjab Kings             Rajasthan Royals                0.5   
6        Deccan Chargers               Delhi Capitals                0.0   
7    Chennai Super Kings               Mumbai Indians                1.0   
8        Deccan Chargers             Rajasthan Royals                0.0   
9           Punjab Kings               Mumbai Indians                0.0   

   team2_recent_form  
0                0.5  
1                0.5  
2                0.5  
3                0.0  
4                0.5  
5     

In [6]:
# Final columns check karo
print("Columns before encoding:")
print(match_df.columns.tolist())
print("\nShape:", match_df.shape)
print("\nSample:")
match_df[['team1', 'team2', 'venue', 'toss_winner', 
          'toss_decision', 'venue_win_rate', 
          'team1_recent_form', 'team2_recent_form', 
          'team1_win']].head()

Columns before encoding:
['match_id', 'team1', 'team2', 'toss_winner', 'toss_decision', 'venue', 'year', 'match_won_by', 'win_outcome', 'team1_win', 'venue_win_rate', 'team1_recent_form', 'team2_recent_form']

Shape: (1169, 13)

Sample:


,team1,team2,venue,toss_winner,toss_decision,venue_win_rate,team1_recent_form,team2_recent_form,team1_win
0,Kolkata Knight Riders,Royal Challengers Bengaluru,M Chinnaswamy Stadium,Royal Challengers Bengaluru,field,0.285714,0.5,0.5,1
1,Chennai Super Kings,Punjab Kings,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,bat,0.500000,0.5,0.5,1
2,Rajasthan Royals,Delhi Capitals,Feroz Shah Kotla,Rajasthan Royals,bat,0.500000,0.5,0.5,0
3,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai Indians,bat,0.611111,0.5,0.0,0
4,Deccan Chargers,Kolkata Knight Riders,Eden Gardens,Deccan Chargers,bat,0.000000,0.5,0.5,0


In [7]:
# Step 1 — Saari unique teams ekta karo
all_teams = sorted(list(set(
    match_df['team1'].tolist() +
    match_df['team2'].tolist()
)))

print("Total unique teams:", len(all_teams))
print(all_teams)

# Step 2 — Teen encoders banao
team_encoder = LabelEncoder()
team_encoder.fit(all_teams)

venue_encoder = LabelEncoder()
venue_encoder.fit(sorted(match_df['venue'].unique()))

toss_encoder = LabelEncoder()
toss_encoder.fit(['bat', 'field'])

# Step 3 — Encoding apply karo
match_df['team1']         = team_encoder.transform(match_df['team1'])
match_df['team2']         = team_encoder.transform(match_df['team2'])
match_df['toss_winner']   = team_encoder.transform(match_df['toss_winner'])
match_df['toss_decision'] = toss_encoder.transform(match_df['toss_decision'])
match_df['venue']         = venue_encoder.transform(match_df['venue'])

print("\nEncoding ke baad sample:")
match_df[['team1','team2','toss_winner','toss_decision','venue','team1_win']].head()

Total unique teams: 15
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

Encoding ke baad sample:


,team1,team2,toss_winner,toss_decision,venue,team1_win
0,6,13,13,1,23,1
1,0,10,0,0,41,1
2,11,2,11,0,16,0
3,8,13,8,0,56,0
4,1,6,1,0,14,0


In [8]:
# Encoders save karo
with open('../model/team_encoder.pkl', 'wb') as f:
    pickle.dump(team_encoder, f)

with open('../model/venue_encoder.pkl', 'wb') as f:
    pickle.dump(venue_encoder, f)

with open('../model/toss_encoder.pkl', 'wb') as f:
    pickle.dump(toss_encoder, f)

# Sirf model ke liye kaam ki columns rakho
final_df = match_df[['team1', 'team2', 'toss_winner', 
                      'toss_decision', 'venue',
                      'venue_win_rate', 'team1_recent_form',
                      'team2_recent_form', 'team1_win']]

# Processed data save karo
final_df.to_csv('../data/processed_matches.csv', index=False)

print("Encoders saved!")
print("Processed data saved!")
print("\nFinal shape:", final_df.shape)
print("\nSample:")
final_df.head()

Encoders saved!
Processed data saved!

Final shape: (1169, 9)

Sample:


,team1,team2,toss_winner,toss_decision,venue,venue_win_rate,team1_recent_form,team2_recent_form,team1_win
0,6,13,13,1,23,0.285714,0.5,0.5,1
1,0,10,0,0,41,0.500000,0.5,0.5,1
2,11,2,11,0,16,0.500000,0.5,0.5,0
3,8,13,8,0,56,0.611111,0.5,0.0,0
4,1,6,1,0,14,0.000000,0.5,0.5,0


In [9]:
print("=" * 50)
print("MODULE 2 SUMMARY — Preprocessing")
print("=" * 50)
print(f"Original dataset    : 283,678 rows (ball-by-ball)")
print(f"Match level         : 1,193 matches")
print(f"After cleaning      : 1,169 matches")
print(f"Removed             : 24 no-result matches")
print(f"Teams               : 15 unique")
print(f"Venues              : 59 unique")
print()
print("STEPS DONE:")
print("1. Ball-by-ball → match level conversion")
print("2. No-result matches remove kiye")
print("3. Team name spelling fix ki — 4 teams")
print("4. Target variable banaya — team1_win")
print("5. Feature Engineering — 3 naye features banaye:")
print("   - venue_win_rate  : team1 ka us venue pe win rate")
print("   - team1_recent_form : last 5 matches ka win rate")
print("   - team2_recent_form : last 5 matches ka win rate")
print("6. LabelEncoder se 3 encoders banaye")
print("7. Encoders pkl mein save kiye")
print("8. Processed CSV save kiya — 9 features")
print()
print("Encoders saved:")
print("  - team_encoder.pkl")
print("  - venue_encoder.pkl")
print("  - toss_encoder.pkl")
print("=" * 50)
print("NEXT: 03_model_training.ipynb")

MODULE 2 SUMMARY — Preprocessing
Original dataset    : 283,678 rows (ball-by-ball)
Match level         : 1,193 matches
After cleaning      : 1,169 matches
Removed             : 24 no-result matches
Teams               : 15 unique
Venues              : 59 unique

STEPS DONE:
1. Ball-by-ball → match level conversion
2. No-result matches remove kiye
3. Team name spelling fix ki — 4 teams
4. Target variable banaya — team1_win
5. Feature Engineering — 3 naye features banaye:
   - venue_win_rate  : team1 ka us venue pe win rate
   - team1_recent_form : last 5 matches ka win rate
   - team2_recent_form : last 5 matches ka win rate
6. LabelEncoder se 3 encoders banaye
7. Encoders pkl mein save kiye
8. Processed CSV save kiya — 9 features

Encoders saved:
  - team_encoder.pkl
  - venue_encoder.pkl
  - toss_encoder.pkl
NEXT: 03_model_training.ipynb
